In [ ]:
import os
import sys
import numpy as np
import pandas as pd
import tensorflow as tf
from pathlib import Path

# 1. Try to detect the local project path explicitly
# This is specific to your local machine setup
local_project_path = '/Users/mirelacertan/Documents/ml_engine'

if os.path.exists(local_project_path):
    os.chdir(local_project_path)
    if local_project_path not in sys.path:
        sys.path.append(local_project_path)
    print(f"Successfully set working directory to: {local_project_path}")
else:
    # Fallback: Check if we are in a 'notebooks' subdir
    if os.getcwd().endswith('notebooks'):
        os.chdir('..')
    
    # Add current directory to path
    if os.getcwd() not in sys.path:
        sys.path.append(os.getcwd())
    print(f"Working directory: {os.getcwd()}")

# 2. Verify we can see the project files
if os.path.exists('train_visual.py'):
    print("Project files detected successfully.")
else:
    print("WARNING: 'train_visual.py' not found in current directory.")
    print("Please ensure you are running this notebook from the project root.")

print(f"TensorFlow Version: {tf.__version__}")
print(f"Devices: {tf.config.list_physical_devices()}")

TensorFlow Version: 2.19.0
Devices: [PhysicalDevice(name='/physical_device:CPU:0', device_type='CPU'), PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [6]:
# Configuration
CONFIG = {
    'model_type': 'tft',
    'multi_task': True,
    'epochs': 100,
    'batch_size': 64,
    'ensemble_size': 3,
    'fetch_data': True,
    'data_dir': 'trained_data/data',
    'checkpoint_dir': 'trained_data/checkpoints/tensorflow',
    'log_dir': 'trained_data/tensorboard',
    'instruments': ['USD_JPY', 'EUR_USD', 'GBP_USD'],
    'candles': 5000
}

print("Configuration loaded successfully.")
print(f"Model: {CONFIG['model_type'].upper()}")
print(f"Instruments: {CONFIG['instruments']}")
print("You can now run the next cell to fetch data.")

Configuration loaded successfully.
Model: TFT
Instruments: ['USD_JPY', 'EUR_USD', 'GBP_USD']
You can now run the next cell to fetch data.


In [8]:
# Fetch Fresh Data (Optional)
import sys
import os

# Ensure project root is in path (in case the first cell wasn't run or path was lost)
if os.getcwd() not in sys.path:
    sys.path.append(os.getcwd())

try:
    from train_visual import fetch_fresh_oanda_data
except ImportError as e:
    print(f"Error importing train_visual: {e}")
    print(f"Current working directory: {os.getcwd()}")
    print("Make sure you are running this notebook from the project root.")
    raise

if 'CONFIG' not in locals():
    print("Error: CONFIG is not defined. Please run the 'Configuration' cell above first.")
else:
    if CONFIG['fetch_data']:
        print("Fetching fresh data from OANDA...")
        try:
            fetch_fresh_oanda_data(
                instruments=CONFIG['instruments'],
                count=CONFIG['candles'],
                output_dir=CONFIG['data_dir']
            )
        except Exception as e:
            print(f"Warning: Could not fetch data ({e}). Using existing files.")

Error importing train_visual: No module named 'train_visual'
Current working directory: /content
Make sure you are running this notebook from the project root.


ModuleNotFoundError: No module named 'train_visual'

In [ ]:
# Load Data
from tensorflow_data_pipeline import load_tensorflow_multitask_data
from utils import load_config

# Find the main data file (prefer USD_JPY)
data_dir = Path(CONFIG['data_dir'])
data_files = list(data_dir.glob("*.csv"))
if not data_files:
    raise FileNotFoundError(f"No CSV files found in {data_dir}")

# Prefer USD_JPY, else take the first one
target_file = next((f for f in data_files if "USD_JPY" in f.name), data_files[0])
print(f"Training on: {target_file}")

# Load config.yaml for feature engineering settings
project_config = load_config('config.yaml')

# Load data as numpy arrays
x_train, y_train, x_val, y_val, x_test, y_test, meta = load_tensorflow_multitask_data(
    str(target_file),
    project_config,
    state_classes=3,
    validation_split=0.2,
    add_all_features=True
)

print(f"Training Samples: {len(x_train)}")
print(f"Validation Samples: {len(x_val)}")
print(f"Input Shape: {x_train.shape}")

In [ ]:
# Build Model
from tensorflow_engine import TensorFlowEngine

# Construct Engine Config
tf_config = {
    'model': {
        'type': CONFIG['model_type'],
        'input_size': x_train.shape[-1],
        'hidden_size': 128,
        'num_layers': 3,
        'dropout': 0.3,
        'num_heads': 4,
        'multi_task': CONFIG['multi_task'],
        'state_classes': 3,
        'sequence_length': x_train.shape[1]
    },
    'optimizer': {
        'type': 'adamw',
        'learning_rate': 0.0003,
        'weight_decay': 0.001
    },
    'training': {
        'epochs': CONFIG['epochs'],
        'early_stopping_patience': 15
    },
    'paths': {
        'checkpoint_dir': CONFIG['checkpoint_dir'],
        'tensorboard_dir': CONFIG['log_dir']
    },
    'mixed_precision': True, # Enable for GPU speedup
    'batch_size': CONFIG['batch_size']
}

# Initialize Engine
engine = TensorFlowEngine(tf_config)
engine.build_model()

# Show Model Summary
engine.model.summary()

In [ ]:
# Train
print("Starting Training...")
history = engine.train(x_train, y_train, x_val, y_val)

In [ ]:
# Save Scalers for Buddy
from tensorflow_data_pipeline import save_scalers

# Save scalers so Buddy can normalize live data correctly
save_scalers(
    {
        'scalers': {
            'feature_scaler': meta.get('feature_scaler'),
            'target_scaler': meta.get('target_scaler')
        },
        'n_features': x_train.shape[-1],
        'sequence_length': x_train.shape[1],
        'state_classes': 3
    },
    CONFIG['checkpoint_dir']
)
print(f"Scalers saved to {CONFIG['checkpoint_dir']}")